<a href="https://colab.research.google.com/github/Arsh-Vora/Arxiv-PageRank-Analysis/blob/dev/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)


100%|██████████| 1.58G/1.58G [00:16<00:00, 104MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/276


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import *

spark = SparkSession.builder.appName("arXiv Metadata").getOrCreate()

In [4]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("authors_parsed", ArrayType(ArrayType(StringType())), True)
])

raw_df = spark.read.json(path + "/arxiv-metadata-oai-snapshot.json", schema=schema)

df_processed = raw_df.withColumn(
    "author_list",
    F.expr("""
        transform(authors_parsed, x ->
            regexp_replace(trim(concat(x[1], ' ', x[0])), "\\\\\\\\[\'`^\\\"~]", "")
        )
    """)

)

authors_exploded = df_processed.select(
    "id",
    F.explode("author_list").alias("author")
)

edges = authors_exploded.alias("a").join(
    authors_exploded.alias("b"), "id"
).filter("a.author != b.author") \
 .select(F.col("a.author").alias("source"), F.col("b.author").alias("target")) \
 .distinct()

edges.cache()
print(f"Graph initialized with {edges.count()}unique edges")

NameError: name 'ArrayType' is not defined